In [ ]:
# pipeline() = easiest way to run a model
# It handles: tokenization → model → decode output
# "sentiment-analysis" = classify if text is positive or negative

from transformers import pipeline


classifier = pipeline("sentiment-analysis")

texts = [
     "I love building AI applications!",
    "This bug is driving me crazy.",
    "The model performance is okay I guess.",
    "RAG systems are incredibly powerful for production apps.",
]

print("=== Sentiment Analysis (running real neural network locally) ===\n")


for text in texts:
    result = classifier(texts)[0]
    label = result['label']
    score = result['score']
    bar = "█"* int(score * 20)
    print(f"Text:  '{text}'")
    print(f"Label: {label} | Confidence: {score:.1%} {bar}")
    print()




# Same pipeline API works for many tasks — just change the task name
print("=== Other tasks you can swap in ===")
print("pipeline('text-generation')     → generate text like GPT")
print("pipeline('summarization')       → summarize long documents")
print("pipeline('translation')         → translate languages")
print("pipeline('question-answering')  → answer from a context (mini RAG!)")
print("pipeline('fill-mask')           → predict missing words in sentence")

In [ ]:
# Cell 16 — Real Embeddings: the actual vectors your RAG app uses
# In Cell 8 we used FAKE vectors. Now we generate REAL ones from text.

# Install: pip install sentence-transformers
# Downloads ~90MB model on first run

from sentence_transformers import SentenceTransformer
import numpy as np

# This model converts any text → 384-dimensional vector
# Same type of model Qdrant/Pinecone use under the hood in your RAG apps
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- Part 1: See what embeddings look like ---
text = "Python is great for machine learning"
embedding = model.encode(text)

print(f"Text: '{text}'")
print(f"Embedding shape: {embedding.shape}")       # (384,) — 384 numbers
print(f"First 8 values: {embedding[:8].round(4)}") # peek at the actual numbers
print(f"Min: {embedding.min():.4f}, Max: {embedding.max():.4f}\n")

In [ ]:
# --- Part 2: Semantic similarity (same cosine sim from Cell 4, real vectors now) ---
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [
    ("I love cats", "I adore kittens"),          # same meaning, different words
    ("I love cats", "Python is a programming language"),  # unrelated
    ("RAG retrieves documents", "vector search finds relevant chunks"),  # related AI concepts
    ("The sky is blue", "Neural networks learn from data"),  # unrelated
]

print("=== Semantic Similarity (real embeddings) ===\n")
for text_a, text_b in pairs:
    vec_a = model.encode(text_a)
    vec_b = model.encode(text_b)
    sim = cosine_similarity(vec_a, vec_b)
    bar = "█" * int(sim * 30)
    print(f"A: '{text_a}'")
    print(f"B: '{text_b}'")
    print(f"Similarity: {sim:.3f} {bar}\n")

In [ ]:
# --- Part 3: Mini RAG — exactly what your Qdrant app does ---
docs = [
    "LangChain helps build LLM applications with chains and agents",
    "The Eiffel Tower is located in Paris, France",
    "RAG combines retrieval with generation to answer from your own data",
    "Cristiano Ronaldo is a famous football player",
    "Vector databases store embeddings for semantic search",
]

query = "how do I build an AI app that searches my documents?"

# Embed all docs + query
doc_embeddings = model.encode(docs)
query_embeddings = model.encode(query)

# Find most similar docs in query
similarities = [cosine_similarity(query_embeddings,doc_emb) for doc_emb in doc_embeddings]

# Sort and show results
ranked = sorted(zip(similarities,docs), reverse=True)


print("=== Mini RAG Search ===")
print(f"Query: '{query}'\n")
print("Ranked results:")
for score, doc in ranked:
    bar = "█" * int(score * 25)
    print(f"  {score:.3f} {bar}")
    print(f"  {doc}\n")

In [ ]:
# Cell 17 — What you built in this notebook (your mental model map)

print("""
ML FUNDAMENTALS — WHAT YOU NOW UNDERSTAND
==========================================

NUMPY (Cell 1-4)
  Arrays → fast math on data
  Cosine similarity → measure direction between vectors
  Matrices → how data is stored in ML

PANDAS (Cell 5-6)
  DataFrames → work with structured data
  Filter/sort/query → explore datasets

MATPLOTLIB (Cell 7)
  Visualize data → bar charts, plots

SCIKIT-LEARN (Cell 9-11)
  LogisticRegression → classify (open/closed)
  LinearRegression   → predict a number
  train_test_split   → never test on training data
  StandardScaler     → scale features or big numbers dominate

NEURAL NETWORK (Cell 12)
  Forward pass  → predict
  Loss          → how wrong
  Backward pass → fix weights (backprop)

TRANSFORMERS (Cell 13)
  Attention → each token sees every other token
  Softmax   → weights must sum to 1.0
  This is the core of GPT, Claude, Llama

TOKENIZATION (Cell 14)
  LLMs read tokens not words
  Rare words = more tokens = higher cost

HUGGING FACE (Cell 15)
  pipeline() → run any open source model in 3 lines
  500k+ models, same API

EMBEDDINGS (Cell 16)
  Text → 384 numbers
  Cosine similarity on real vectors
  This is exactly what your RAG apps do
""")